## Build an AI Agent with Microsoft Agent Framework

### Expense Claim Agent

This notebook implements the Microsoft Learn lab: an AI agent reads expense data and uses a custom **tool** function to "send" an itemized expense-claim email.

### Installing Required Libraries

In [ ]:
%pip install agent-framework-core agent-framework-foundry python-dotenv

### Add references

In [3]:
import os
from dotenv import load_dotenv

from agent_framework import tool, Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential
from pydantic import Field
from typing import Annotated

### Setting up the Environment

In [4]:
load_dotenv()
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME")

print("Project Endpoint:", project_endpoint)
print("Model:", model)

Project Endpoint: https://udemo-103-resource.services.ai.azure.com/api/projects/udemo-103
Model: gpt-5.2


### Create a tool function for the email functionality

Tools are a way to add custom functionality to agents. This function *simulates* sending an email by printing it to the console. In a real application, you'd use an SMTP service or similar to actually send the email.

In [5]:
# Create a tool function for the email functionality
@tool(approval_mode="never_require")
def submit_claim(
    to: Annotated[str, Field(description="Who to send the email to")],
    subject: Annotated[str, Field(description="The subject of the email.")],
    body: Annotated[str, Field(description="The text body of the email.")]):
        print("\nTo:", to)
        print("Subject:", subject)
        print(body, "\n")

### Create the agent and process the expenses data

The `process_expenses_data` function creates a Foundry chat client, initializes the agent with the tool and instructions, and then invokes the agent with the expenses data.

In [6]:
async def process_expenses_data(prompt, expenses_data):
    # Create a foundry chat client
    client = FoundryChatClient(
        project_endpoint=project_endpoint,
        model=model,
        credential=DefaultAzureCredential(),
    )

    # Initialize an agent with the tool and instructions
    async with (
        Agent(
            client=client,
            name="ExpenseClaimAgent",
            instructions="""You are an AI assistant for expense claim submission.
                        At the user's request, create an expense claim and use the plug-in function to send an email to expenses@contoso.com with the subject 'Expense Claim' and a body that contains itemized expenses with a total.
                        Then confirm to the user that you've done so. Don't ask for any more information from the user, just use the data provided to create the email.""",
            tools=[submit_claim],
        ) as agent,
    ):
        # Use the agent to process the expenses data
        try:
            # Add the input prompt to a list of messages to be submitted
            prompt_messages = [f"{prompt}: {expenses_data}"]
            # Invoke the agent with the messages
            response = await agent.run(prompt_messages)
            # Display the response
            print(f"\n# Agent:\n{response}")
        except Exception as e:
            # Something went wrong
            print(e)

### Provide the expenses data and run the agent

In [7]:
# Sample expenses data (in the lab this is read from a data file)
expenses_data = """date,description,amount
07-Mar-2025,taxi,24.00
07-Mar-2025,dinner,65.50
07-Mar-2025,hotel,125.90"""

prompt = "Submit an expense claim"

await process_expenses_data(prompt, expenses_data)


To: expenses@contoso.com
Subject: Expense Claim
Expense Claim

Itemized Expenses:
- 07-Mar-2025 | taxi  | $24.00
- 07-Mar-2025 | dinner | $65.50
- 07-Mar-2025 | hotel | $125.90

Total: $215.40 


# Agent:
I’ve submitted your expense claim to **expenses@contoso.com** with the subject **“Expense Claim”**.

**Itemized expenses**
- 07-Mar-2025 — taxi — $24.00  
- 07-Mar-2025 — dinner — $65.50  
- 07-Mar-2025 — hotel — $125.90  

**Total: $215.40**
